# Restaurant Agent With LLM And Gradio

In the previous notebook, our restaurant agent followed fixed Python rules.

Now we will build a similar restaurant agent, but one node will use an LLM.

Then we will make a small Gradio app so a user can type an order and get a response.

OpenAI is paid, so this notebook tries to use Groq first with `GROQ_API_KEY`. If you later add `OPENAI_API_KEY`, you can use OpenAI too.

## Big Picture In Easy Words

Imagine a restaurant.

A customer says: I want biryani.

The restaurant process is:

1. Waiter takes the order
2. Kitchen reads the order
3. Chef thinks about what to cook
4. Waiter replies to the customer

In our agent:

| Restaurant Thing | Agent Thing |
| --- | --- |
| Order paper | State |
| Waiter | Node |
| Chef thinking | LLM node |
| Path between workers | Edge |
| Full restaurant process | Graph |

## Free-Friendly LLM Choice

OpenAI works well, but it normally needs paid billing.

For learning, use one of these:

1. Groq API: good for this notebook. Add `GROQ_API_KEY` in `.env`.
2. Ollama: runs locally on your laptop, no API key, but you must install Ollama and download a model.
3. Gemini/Groq free tiers can change, so always check their current dashboard before depending on them.

This notebook uses Groq first because your environment already has `langchain-groq` installed.

## Step 1: Import Libraries

`dotenv` loads keys from `.env`.

`StateGraph`, `START`, and `END` build the LangGraph workflow.

`add_messages` keeps message history.

`ChatGroq` is the free-friendly LLM option here.

`ChatOpenAI` is optional if you have an OpenAI key.

`gradio` creates the web app.

In [ ]:
import os
from typing import Annotated, TypedDict

import gradio as gr
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

## Step 2: Load API Keys

The `.env` file is like a private notebook for secret keys.

Never upload `.env` to GitHub.

For Groq, put this in `.env`:

`GROQ_API_KEY=your_key_here`

For OpenAI, put this in `.env` only if you want to use paid OpenAI:

`OPENAI_API_KEY=your_key_here`

In [ ]:
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

print("GROQ_API_KEY:", "set" if groq_key else "missing")
print("OPENAI_API_KEY:", "set" if openai_key else "missing")

## Step 3: Create The LLM

This function chooses which LLM to use.

First choice: Groq, because it is better for learning without OpenAI billing.

Second choice: OpenAI, only if you added an OpenAI key.

If no key exists, the notebook gives a clear message instead of confusing errors.

In [ ]:
def create_llm():
    if os.getenv("GROQ_API_KEY"):
        print("Using Groq LLM")
        return ChatGroq(model="llama-3.1-8b-instant", temperature=0.3)

    if os.getenv("OPENAI_API_KEY"):
        print("Using OpenAI LLM")
        return ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

    print("No LLM key found. Add GROQ_API_KEY in .env for the free-friendly option.")
    return None


llm = create_llm()

## Step 4: Define State

State is the information bag that travels through the graph.

For the restaurant, state carries:

- `order`: what the customer wants
- `messages`: history of what happened
- `kitchen_plan`: the LLM answer from the chef node
- `final_response`: the final message shown to the customer

In [ ]:
class LLRestaurantState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    order: str
    kitchen_plan: str
    final_response: str

## Step 5: Create Nodes

We will create three nodes.

1. `take_order`: receives the customer order
2. `chef_llm`: asks the LLM to think like a helpful restaurant chef
3. `serve_customer`: prepares the final response

Only `chef_llm` uses the LLM. The other nodes are simple Python.

In [ ]:
def take_order(state: LLRestaurantState) -> LLRestaurantState:
    order = state.get("order", "biryani")
    print("1. Waiter received order:", order)

    return {
        "order": order,
        "messages": [f"Customer ordered: {order}"]
    }


def chef_llm(state: LLRestaurantState) -> LLRestaurantState:
    if llm is None:
        raise ValueError("Please add GROQ_API_KEY in .env, then restart the notebook kernel.")

    order = state["order"]
    print("2. Chef LLM is thinking about:", order)

    system_prompt = """
You are a friendly restaurant chef assistant.
Reply in very simple English.
Keep the answer short.
Tell the customer that the order is accepted and what will happen next.
"""

    user_prompt = f"Customer ordered: {order}. Create a short restaurant reply."

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])

    return {
        "kitchen_plan": response.content,
        "messages": [response]
    }


def serve_customer(state: LLRestaurantState) -> LLRestaurantState:
    print("3. Waiter is serving the LLM response")

    final_response = state["kitchen_plan"]

    return {
        "final_response": final_response,
        "messages": [f"Final response sent to customer: {final_response}"]
    }

## Step 6: Build The Graph

Now we connect the nodes with edges.

The flow is:

`START -> take_order -> chef_llm -> serve_customer -> END`

This is like a restaurant route:

Customer -> Waiter -> Chef -> Waiter -> Customer

In [ ]:
llm_restaurant_builder = StateGraph(LLRestaurantState)

llm_restaurant_builder.add_node("take_order", take_order)
llm_restaurant_builder.add_node("chef_llm", chef_llm)
llm_restaurant_builder.add_node("serve_customer", serve_customer)

llm_restaurant_builder.add_edge(START, "take_order")
llm_restaurant_builder.add_edge("take_order", "chef_llm")
llm_restaurant_builder.add_edge("chef_llm", "serve_customer")
llm_restaurant_builder.add_edge("serve_customer", END)

llm_restaurant_graph = llm_restaurant_builder.compile()

## Step 7: Display The Graph

Run this cell to see the graph object or a small diagram.

In [ ]:
llm_restaurant_graph

## Step 8: Run The Agent Once

This cell sends one order to the agent.

If you do not have `GROQ_API_KEY` or `OPENAI_API_KEY`, this cell will ask you to add a key first.

In [ ]:
result = llm_restaurant_graph.invoke({
    "order": "biryani",
    "messages": []
})

result["final_response"]

## Step 9: Create A Gradio App

Gradio makes a simple web app from a Python function.

The user types an order.

Our function sends that order into the LangGraph agent.

The agent returns the final LLM response.

In [ ]:
def restaurant_agent_app(order: str) -> str:
    if not order.strip():
        return "Please enter an order, for example: biryani."

    try:
        result = llm_restaurant_graph.invoke({
            "order": order,
            "messages": []
        })
        return result["final_response"]
    except Exception as error:
        return f"Error: {error}"

## Step 10: Launch The Gradio App

Run this cell to start the app.

A local URL will appear below the cell.

Open that URL in your browser and type a restaurant order.

When you are done, stop the cell or interrupt the kernel.

In [ ]:
demo = gr.Interface(
    fn=restaurant_agent_app,
    inputs=gr.Textbox(label="Customer Order", placeholder="Example: I want chicken biryani"),
    outputs=gr.Textbox(label="Restaurant Agent Reply"),
    title="Restaurant LLM Agent",
    description="A simple LangGraph agent with one LLM node."
)

demo.launch()

## What Did We Build?

We built a basic LLM agent.

The agent is not only one direct LLM call. It has a structure:

1. State carries the order
2. First node receives the order
3. LLM node thinks and creates a reply
4. Final node serves the reply
5. Gradio makes it usable as an app

This is the same idea as the restaurant example, but now the chef node can think using an LLM.

## Example LLM Output

Because this notebook uses an LLM, the exact words may change each time.

For an order like `I want chicken biryani`, the app may reply something like:

```text
Great choice! Your chicken biryani order is accepted. The kitchen will start preparing it now.
```

The answer is short and restaurant-focused because the system prompt tells the LLM to act like a friendly restaurant chef assistant.

# Bonus: Agent That Chooses Different Routes

Now we will add the idea you asked about:

`refund`, `cook_food`, `ask_question`, `call_tool`, or `end_chat`.

This is where LangGraph is very helpful.

Instead of always going in one straight line, the graph can choose a different path depending on the user message.

## Easy Daily Life Example

Imagine a restaurant help desk.

Different customers say different things:

- I want biryani -> cook food
- I want my money back -> refund
- What is available? -> ask a question / clarify
- Check my order status -> call a tool
- Bye -> end chat

A normal Python program can do this with many `if` statements.

LangGraph makes these routes clear as a graph.

## Define Routing State

This state is like a customer support form.

It stores:

- `user_message`: what the customer said
- `route`: where the agent should go next
- `final_response`: answer returned to the customer

In [ ]:
class RoutingState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    user_message: str
    route: str
    final_response: str

## Create The Decision Node

This node decides which path to take.

For learning, we use simple keyword rules.

Later, you can replace this with an LLM decision.

In [ ]:
def decide_route(state: RoutingState) -> RoutingState:
    user_message = state["user_message"].lower()

    if "refund" in user_message or "money back" in user_message:
        route = "refund"
    elif "status" in user_message or "track" in user_message or "check" in user_message:
        route = "call_tool"
    elif "?" in user_message or "available" in user_message or "menu" in user_message:
        route = "ask_question"
    elif "bye" in user_message or "exit" in user_message or "stop" in user_message:
        route = "end_chat"
    else:
        route = "cook_food"

    print("Selected route:", route)
    return {"route": route, "messages": [f"Router selected: {route}"]}

## Create Action Nodes

Each node handles one route.

This is like sending the customer to the correct counter.

In [ ]:
def refund_node(state: RoutingState) -> RoutingState:
    response = "I understand. I will start your refund request. Please share your order number."
    return {"final_response": response, "messages": [response]}


def cook_food_node(state: RoutingState) -> RoutingState:
    response = "Your order is accepted. The kitchen will start preparing your food now."
    return {"final_response": response, "messages": [response]}


def ask_question_node(state: RoutingState) -> RoutingState:
    response = "Sure. Please tell me which item you want, for example biryani, burger, or pizza."
    return {"final_response": response, "messages": [response]}


def call_tool_node(state: RoutingState) -> RoutingState:
    # This is a fake tool for learning. Later this could call a real database or API.
    fake_order_status = "Your order is being prepared and will be ready in 15 minutes."
    return {"final_response": fake_order_status, "messages": [fake_order_status]}


def end_chat_node(state: RoutingState) -> RoutingState:
    response = "Thank you. Chat ended. Have a nice day!"
    return {"final_response": response, "messages": [response]}

## Routing Function

This function tells LangGraph which edge to follow.

It reads `state["route"]` and sends the graph to the matching node.

In [ ]:
def route_after_decision(state: RoutingState) -> str:
    return state["route"]

## Build Conditional Graph

This graph does not always go to the same node.

It starts at `decide_route`.

Then LangGraph chooses one route:

- `refund`
- `cook_food`
- `ask_question`
- `call_tool`
- `end_chat`

In [ ]:
routing_builder = StateGraph(RoutingState)

routing_builder.add_node("decide_route", decide_route)
routing_builder.add_node("refund", refund_node)
routing_builder.add_node("cook_food", cook_food_node)
routing_builder.add_node("ask_question", ask_question_node)
routing_builder.add_node("call_tool", call_tool_node)
routing_builder.add_node("end_chat", end_chat_node)

routing_builder.add_edge(START, "decide_route")

routing_builder.add_conditional_edges(
    "decide_route",
    route_after_decision,
    {
        "refund": "refund",
        "cook_food": "cook_food",
        "ask_question": "ask_question",
        "call_tool": "call_tool",
        "end_chat": "end_chat",
    }
)

routing_builder.add_edge("refund", END)
routing_builder.add_edge("cook_food", END)
routing_builder.add_edge("ask_question", END)
routing_builder.add_edge("call_tool", END)
routing_builder.add_edge("end_chat", END)

routing_graph = routing_builder.compile()

## Test Different Messages

Run these examples one by one.

You will see the graph choose different paths.

In [ ]:
examples = [
    "I want chicken biryani",
    "I want a refund",
    "What is available on the menu?",
    "Please check my order status",
    "bye"
]

for text in examples:
    print("User:", text)
    output = routing_graph.invoke({"user_message": text, "messages": []})
    print("Agent:", output["final_response"])
    print("-" * 50)

## Gradio App For Routing Agent

This app lets you type any message.

LangGraph chooses the correct route and returns the answer.

In [ ]:
def routing_agent_app(user_message: str) -> str:
    if not user_message.strip():
        return "Please type a message."

    output = routing_graph.invoke({
        "user_message": user_message,
        "messages": []
    })

    return output["final_response"]

In [ ]:
routing_demo = gr.Interface(
    fn=routing_agent_app,
    inputs=gr.Textbox(label="Customer Message", placeholder="Try: I want a refund"),
    outputs=gr.Textbox(label="Agent Reply"),
    title="Restaurant Routing Agent",
    description="LangGraph chooses between refund, cook_food, ask_question, call_tool, and end_chat."
)

routing_demo.launch()